# Automatic Context Compaction

장시간 실행되는 agentic 작업은 종종 context 한도를 초과합니다. Tool을 많이 사용하는 workflow나 긴 대화는 token context window를 빠르게 소모합니다. [Effective Context Engineering for AI Agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)에서는 context 관리가 성능 저하와 context rot을 방지하는 데 어떻게 도움이 되는지 논의했습니다.

Claude Agent Python SDK는 token 사용량이 설정 가능한 threshold를 초과할 때 대화 기록을 자동으로 압축하여 이 context를 관리하고, 일반적인 200k token context 한도를 넘어서도 작업을 계속할 수 있게 합니다.

이 cookbook에서는 **agentic 고객 서비스 workflow**를 통해 context compaction을 시연하겠습니다. 지원 ticket queue를 처리하는 AI 고객 서비스 agent를 구축했다고 상상해 보세요. 각 ticket에 대해 문제를 분류하고, knowledge base를 검색하고, 우선순위를 설정하고, 적절한 팀으로 전달하고, 응답 초안을 작성하고, 완료로 표시해야 합니다. Ticket을 하나씩 처리하면서 대화 기록에는 분류 결과, knowledge base 검색, 응답 초안이 쌓여 수천 token을 빠르게 소모합니다.

## Context Compaction이란?

Tool use를 사용하는 agentic workflow를 구축할 때 agent가 복잡한 작업을 반복 수행하면 대화가 매우 커질 수 있습니다. `compaction_control` parameter는 다음과 같이 context를 자동 관리합니다.

1. 대화의 각 turn에서 token 사용량을 모니터링합니다.
2. Threshold를 초과하면 요약 prompt를 user turn으로 삽입합니다.
3. 모델이 `<summary></summary>` tag로 감싼 요약을 생성하게 합니다. 이 tag는 parsing되지 않지만 모델을 유도하는 데 도움이 됩니다.
4. 대화 기록을 지우고 요약만으로 재개합니다.
5. 압축된 context로 작업을 계속합니다.

## 이 cookbook을 마치면 다음을 할 수 있습니다.
 
 - 반복적인 workflow에서 context 한도를 효과적으로 관리하는 방법 이해하기
 - automatic context compaction을 활용하는 agent 작성하기
 - 여러 반복에 걸쳐 초점을 유지하는 workflow 설계하기

## 사전 요구사항

이 guide를 따라 하기 전에 다음을 준비하세요.

**필요한 지식**

- agentic 패턴과 tool calling에 대한 기본 이해

**필요한 도구**

- Python 3.11 이상
- Anthropic API key
- Anthropic SDK >= 0.74.1


> **Opus 4.6을 사용하시나요?** SDK 수준 설정 없이 context 관리를 자동으로 처리하는 [server-side compaction](https://docs.anthropic.com/en/docs/build-with-claude/compaction)을 권장합니다.
>
> 이 cookbook은 이전 모델을 사용하거나 요약에 다른 (더 저렴한) 모델을 사용하려는 경우 유용한 **SDK 기반 compaction**을 다룹니다.


## 설정

먼저 필요한 dependencies를 설치합니다.


In [1]:
# %pip install -qU anthropic python-dotenv

참고: .env 파일에 다음이 포함되어 있는지 확인합니다.

`ANTHROPIC_API_KEY=your_key_here`

환경 변수를 불러오고 client를 설정합니다. Claude message 응답을 시각화하는 helper utility도 불러옵니다.



In [2]:
from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-sonnet-4-6"

## 준비하기

[utils/customer_service_tools.py](utils/customer_service_tools.py)에 고객 지원 ticket을 처리하는 여러 함수를 정의했습니다.

- `get_next_ticket()` - queue에서 다음 미처리 ticket을 가져옵니다.
- `classify_ticket(ticket_id, category)` - 문제를 billing, technical, account, product, shipping으로 분류합니다.
- `search_knowledge_base(query)` - 관련 도움말과 해결책을 찾습니다.
- `set_priority(ticket_id, priority)` - 우선순위 수준(low, medium, high, urgent)을 지정합니다.
- `route_to_team(ticket_id, team)` - ticket을 적절한 지원 팀으로 전달합니다.
- `draft_response(ticket_id, response_text)` - 고객에게 보낼 응답을 만듭니다.
- `mark_complete(ticket_id)` - 처리된 ticket을 마무리합니다.

고객 서비스 agent는 이러한 tool로 ticket을 체계적으로 처리할 수 있습니다. 각 ticket에는 분류, 조사, 우선순위 지정, 전달, 응답 초안 작성이 필요합니다. 20~30개 ticket을 순서대로 처리하면 모든 분류, knowledge base 검색, 응답 초안의 tool 결과가 대화 기록에 쌓여 token이 선형으로 증가합니다.

Tool에 `beta_tool` decorator를 사용하여 Claude agent가 접근할 수 있게 합니다. Decorator는 함수 argument와 docstring을 추출하여 Claude에 tool metadata로 제공합니다.

```python
import anthropic
from anthropic import beta_tool

@beta_tool
def get_next_ticket() -> dict:
    """Queue에서 다음 미처리 지원 ticket을 가져옵니다."""
    ...
```



In [3]:
import anthropic
from utils.customer_service_tools import (
    classify_ticket,
    draft_response,
    get_next_ticket,
    initialize_ticket_queue,
    mark_complete,
    route_to_team,
    search_knowledge_base,
    set_priority,
)

client = anthropic.Anthropic()

tools = [
    get_next_ticket,
    classify_ticket,
    search_knowledge_base,
    set_priority,
    route_to_team,
    draft_response,
    mark_complete,
]

## Baseline: Compaction 없이 실행하기

현실적인 고객 서비스 시나리오인 지원 ticket queue 처리부터 시작하겠습니다.

Workflow는 다음과 같습니다.

**각 Ticket에 대해:**
1. `get_next_ticket()`으로 ticket을 가져옵니다.
2. 문제 범주를 분류합니다(billing, technical, account, product, shipping).
3. Knowledge base에서 관련 정보를 검색합니다.
4. 적절한 우선순위를 설정합니다(low, medium, high, urgent).
5. 올바른 팀으로 전달합니다.
6. 고객 응답 초안을 작성합니다.
7. Ticket을 완료로 표시합니다.
8. 다음 ticket으로 이동합니다.

**어려움**: Queue에 ticket 5개가 있고 각 ticket에 7번의 tool 호출이 필요하면 Claude는 35번 이상의 tool 호출을 수행합니다. 분류, knowledge base 검색, 응답 초안을 포함한 각 단계의 결과가 대화 기록에 쌓입니다. Compaction 없이는 모든 ticket에 대해 이 데이터가 모두 memory에 남으며, 5번 ticket에 이르면 context에 앞선 ticket 4개의 전체 세부사항이 포함됩니다.

먼저 **compaction 없이** 이 workflow를 실행하고 어떻게 되는지 살펴보겠습니다.


In [4]:
from anthropic.types.beta import BetaMessageParam

num_tickets = 5
initialize_ticket_queue(num_tickets)

messages: list[BetaMessageParam] = [
    {
        "role": "user",
        "content": f"""You are an AI customer service agent. Your task is to process support tickets from a queue.

For EACH ticket, you must complete ALL these steps:

1. **Fetch ticket**: Call get_next_ticket() to retrieve the next unprocessed ticket
2. **Classify**: Call classify_ticket() to categorize the issue (billing/technical/account/product/shipping)
3. **Research**: Call search_knowledge_base() to find relevant information for this ticket type
4. **Prioritize**: Call set_priority() to assign priority (low/medium/high/urgent) based on severity
5. **Route**: Call route_to_team() to assign to the appropriate team
6. **Draft**: Call draft_response() to create a helpful customer response using KB information
7. **Complete**: Call mark_complete() to finalize this ticket
8. **Continue**: Immediately fetch the next ticket and repeat

IMPORTANT RULES:
- Process tickets ONE AT A TIME in sequence
- Complete ALL 7 steps for each ticket before moving to the next
- Keep fetching and processing tickets until you get an error that the queue is empty
- There are {num_tickets} tickets total - process all of them
- Be thorough but efficient

Begin by fetching the first ticket.""",
    }
]

total_input = 0
total_output = 0
turn_count = 0

runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=4096,
    tools=tools,
    messages=messages,
)

for message in runner:
    messages_list = list(runner._params["messages"])
    turn_count += 1
    total_input += message.usage.input_tokens
    total_output += message.usage.output_tokens
    print(
        f"Turn {turn_count:2d}: Input={message.usage.input_tokens:7,} tokens | "
        f"Output={message.usage.output_tokens:5,} tokens | "
        f"Messages={len(messages_list):2d} | "
        f"Cumulative In={total_input:8,}"
    )

print(f"\n{'=' * 60}")
print("BASELINE RESULTS (NO COMPACTION)")
print(f"{'=' * 60}")
print(f"Total turns:   {turn_count}")
print(f"Input tokens:  {total_input:,}")
print(f"Output tokens: {total_output:,}")
print(f"Total tokens:  {total_input + total_output:,}")
print(f"{'=' * 60}")

Turn  1: Input=  1,537 tokens | Output=   57 tokens | Messages= 1 | Cumulative In=   1,537
Turn  2: Input=  1,760 tokens | Output=  102 tokens | Messages= 3 | Cumulative In=   3,297
Turn  3: Input=  1,905 tokens | Output=   88 tokens | Messages= 5 | Cumulative In=   5,202
Turn  4: Input=  2,237 tokens | Output=   84 tokens | Messages= 7 | Cumulative In=   7,439
Turn  5: Input=  2,385 tokens | Output=   89 tokens | Messages= 9 | Cumulative In=   9,824
Turn  6: Input=  2,537 tokens | Output=  301 tokens | Messages=11 | Cumulative In=  12,361
Turn  7: Input=  2,888 tokens | Output=   67 tokens | Messages=13 | Cumulative In=  15,249
Turn  8: Input=  3,079 tokens | Output=   56 tokens | Messages=15 | Cumulative In=  18,328
Turn  9: Input=  3,316 tokens | Output=   91 tokens | Messages=17 | Cumulative In=  21,644
Turn 10: Input=  3,450 tokens | Output=   84 tokens | Messages=19 | Cumulative In=  25,094
Turn 11: Input=  3,777 tokens | Output=   84 tokens | Messages=21 | Cumulative In=  28,871

Baseline을 확보했으므로 compaction 없이 context가 어떻게 증가하는지 더 잘 알 수 있습니다. 보시는 것처럼 각 turn마다 입력에 더 많은 token이 추가되어 token이 선형으로 증가합니다.

이는 높은 token 소모로 이어지고 context 한도에 빠르게 도달할 수 있습니다. 27번째 turn까지 ticket 5개만으로 누적 input token이 150,000개입니다.

Compaction 없이 ticket 5개를 모두 처리한 후 Claude의 최종 응답을 살펴보겠습니다.


In [5]:
print(message.content[-1].text)

---

## ✅ ALL TICKETS PROCESSED SUCCESSFULLY!

**Summary of Completed Work:**

I have successfully processed all 5 tickets from the queue. Here's what was accomplished:

1. **TICKET-1** - Sam Smith - Payment method update error
   - Category: Billing | Priority: High | Team: billing-team
   
2. **TICKET-2** - Morgan Johnson - Missing delivery
   - Category: Shipping | Priority: High | Team: logistics-team
   
3. **TICKET-3** - Morgan Jones - Email address change request
   - Category: Account | Priority: Medium | Team: account-services
   
4. **TICKET-4** - Alex Johnson - Wrong item delivered
   - Category: Shipping | Priority: High | Team: logistics-team
   
5. **TICKET-5** - Morgan Jones - Refund request for cancelled subscription
   - Category: Billing | Priority: High | Team: billing-team

Each ticket was:
✅ Classified correctly
✅ Researched in the knowledge base
✅ Assigned appropriate priority
✅ Routed to the correct team
✅ Given a detailed, helpful customer response
✅ Marked as c

### 문제 이해하기

위 baseline workflow에서 Claude는 다음을 해야 했습니다.
- **지원 ticket 5개**를 순서대로 처리
- **Ticket당 7단계** 완료(가져오기, 분류, 조사, 우선순위 지정, 전달, 초안 작성, 완료)
- 결과가 대화 기록에 쌓이는 **tool 호출 35번** 수행
- **모든 분류, knowledge base 검색, 응답 초안**을 memory에 저장

**이렇게 되는 이유:**
1. **선형 token 증가** - Tool을 사용할 때마다 전체 대화 기록(이전의 모든 tool 결과 포함)이 Claude에 전송됩니다.
2. **Context pollution** - Ticket B를 처리하는 동안 ticket A의 분류와 응답 초안이 context에 남아 있습니다.
3. **비용 누적** - 5번 ticket에 이르면 모든 API 호출마다 앞선 ticket 4개의 데이터를 보냅니다.
4. **더 느린 응답** - 방대한 context를 처리하는 데 더 오래 걸립니다.
5. **한도 도달 위험** - 결국 200k token context window에 도달합니다.


**실제로 필요한 것**: Ticket A를 완료한 뒤에는 전체 분류 결과, knowledge base 검색, 완전한 응답 초안 대신 **간단한 요약**(ticket 해결 여부, 범주, 우선순위)만 필요합니다. 상세 workflow는 버리고 완료 요약만 남겨야 합니다.

Automatic context compaction이 이 문제를 어떻게 해결하는지 살펴보겠습니다.


## Automatic Context Compaction 활성화

Automatic context compaction을 활성화하여 정확히 같은 고객 서비스 workflow를 실행하겠습니다. Tool runner에 `compaction_control` parameter만 추가하면 됩니다.

`compaction_control` parameter에는 필수 field 하나와 여러 선택 field가 있습니다.

- **`enabled`** (필수): compaction을 켜고 끄는 Boolean
- **`context_token_threshold`** (선택): compaction을 유발하는 token 수(기본값: 100,000)
- **`model`** (선택): 요약에 사용할 모델(기본값은 주 모델)
- **`summary_prompt`** (선택): 요약 생성을 위한 사용자 정의 prompt

이 고객 서비스 workflow에서는 **5,000 token threshold**를 사용합니다. 즉, 여러 ticket을 처리한 후 compaction이 자동 실행됩니다. 그러면 Claude는 다음을 할 수 있습니다.
1. **완료 요약 유지**(해결된 ticket, 범주, 결과)
2. **상세 tool 결과 버리기**(전체 KB 문서, 완전한 분류, 응답 초안 텍스트)
3. 다음 ticket 묶음을 처리할 때 **새로 시작하기**

이는 실제 지원 agent가 일하는 방식을 모방합니다. Ticket을 해결하고 간단히 기록한 뒤 다음 사례로 넘어갑니다.


In [6]:
# Queue를 다시 초기화하고 compaction을 사용하여 실행
initialize_ticket_queue(num_tickets)

total_input_compact = 0
total_output_compact = 0
turn_count_compact = 0
compaction_count = 0
prev_msg_count = 0

runner = client.beta.messages.tool_runner(
    model=MODEL,
    max_tokens=4096,
    tools=tools,
    messages=messages,
    compaction_control={
        "enabled": True,
        "context_token_threshold": 5000,
    },
)

for message in runner:
    turn_count_compact += 1
    total_input_compact += message.usage.input_tokens
    total_output_compact += message.usage.output_tokens
    messages_list = list(runner._params["messages"])
    curr_msg_count = len(messages_list)

    if curr_msg_count < prev_msg_count:
        # Message 수가 감소하면 compaction을 식별할 수 있습니다
        compaction_count += 1

        print(f"\n{'=' * 60}")
        print(f"🔄 Compaction occurred! Messages: {prev_msg_count} → {curr_msg_count}")
        print("   Summary message after compaction:")
        print(messages_list[-1]["content"][-1].text)  # type: ignore
        print(f"\n{'=' * 60}")

    prev_msg_count = curr_msg_count
    print(
        f"Turn {turn_count_compact:2d}: Input={message.usage.input_tokens:7,} tokens | "
        f"Output={message.usage.output_tokens:5,} tokens | "
        f"Messages={len(messages_list):2d} | "
        f"Cumulative In={total_input_compact:8,}"
    )

print(f"\n{'=' * 60}")
print("OPTIMIZED RESULTS (WITH COMPACTION)")
print(f"{'=' * 60}")
print(f"Total turns:   {turn_count_compact}")
print(f"Compactions:   {compaction_count}")
print(f"Input tokens:  {total_input_compact:,}")
print(f"Output tokens: {total_output_compact:,}")
print(f"Total tokens:  {total_input_compact + total_output_compact:,}")
print(f"{'=' * 60}")

Turn  1: Input=  1,537 tokens | Output=   57 tokens | Messages= 1 | Cumulative In=   1,537
Turn  2: Input=  1,755 tokens | Output=  108 tokens | Messages= 3 | Cumulative In=   3,292
Turn  3: Input=  1,906 tokens | Output=   88 tokens | Messages= 5 | Cumulative In=   5,198
Turn  4: Input=  2,216 tokens | Output=   84 tokens | Messages= 7 | Cumulative In=   7,414
Turn  5: Input=  2,364 tokens | Output=   89 tokens | Messages= 9 | Cumulative In=   9,778
Turn  6: Input=  2,516 tokens | Output=  332 tokens | Messages=11 | Cumulative In=  12,294
Turn  7: Input=  2,898 tokens | Output=   67 tokens | Messages=13 | Cumulative In=  15,192
Turn  8: Input=  3,090 tokens | Output=   56 tokens | Messages=15 | Cumulative In=  18,282
Turn  9: Input=  3,325 tokens | Output=   97 tokens | Messages=17 | Cumulative In=  21,607
Turn 10: Input=  3,465 tokens | Output=   90 tokens | Messages=19 | Cumulative In=  25,072
Turn 11: Input=  3,801 tokens | Output=   84 tokens | Messages=21 | Cumulative In=  28,873

Automatic context compaction을 활성화하면 turn당 token 사용량이 선형으로 증가하지 않고 각 compaction event 후 감소하는 것을 볼 수 있습니다. Ticket 처리 중 compaction event가 두 번 있었고, 뒤따르는 turn에서 전체 token 사용량이 줄어듭니다.

Baseline 버전과 비교하여 token을 79,000개만 사용했습니다. 각 compaction event 후 생성된 요약 message도 출력하여, Claude가 이전 ticket 세부사항을 효과적으로 요약한 모습을 보여줍니다.

Compaction을 활성화하여 ticket 5개를 모두 처리한 후 최종 응답을 살펴보겠습니다.


In [7]:
print(message.content[-1].text)

Perfect! **ALL 5 TICKETS HAVE BEEN SUCCESSFULLY COMPLETED!** 🎉

## Final Summary - All Tickets Processed

### TICKET-5 (Morgan Brown) - **COMPLETED** ✓
- **Issue**: Damaged package (Order #ORD-43312), broken product inside, needs replacement
- **Category**: shipping
- **Priority**: high
- **Team**: logistics-team
- **Status**: resolved
- **Response**: Apologized for damaged shipment, escalated to Logistics Team with HIGH priority, explained they'll process immediate replacement, provide return instructions, and contact customer with tracking and timeline

---

## 🎯 ALL 5 TICKETS COMPLETED

1. ✅ **TICKET-1** (Chris Davis) - Account locked → account-services
2. ✅ **TICKET-2** (Chris Williams) - Billing charge → billing-team  
3. ✅ **TICKET-3** (John Jones) - Google Sheets integration → product-success
4. ✅ **TICKET-4** (Sam Johnson) - Plan comparison → product-success
5. ✅ **TICKET-5** (Morgan Brown) - Damaged shipment → logistics-team

### Processing Statistics
- **Total tickets process

### 결과 비교

Compaction을 활성화하면 workflow와 최종 요약의 품질을 보존하면서 두 실행 사이의 token 절감 차이를 명확히 볼 수 있습니다.

Automatic context compaction으로 다음이 달라졌습니다.

1. **여러 ticket 후 context 초기화** - Ticket 5~7개를 처리하여 5k 이상의 tool 결과 token이 생성되면 SDK는 자동으로:
   - 요약 prompt를 삽입합니다.
   - Claude가 `<summary></summary>` tag로 감싼 완료 요약을 생성하게 합니다.
   - 대화 기록을 지우고 상세 분류, KB 검색, 응답을 버립니다.
   - 완료 요약만으로 계속합니다.

2. **Input token이 한도 내에 유지됨** - Ticket을 더 처리하면서 100k 이상으로 누적되는 대신, input token이 각 compaction 후 초기화됩니다. 5번 ticket을 처리할 때 1~4번 ticket의 전체 tool 결과를 갖고 있지 않습니다.

3. **작업이 성공적으로 완료됨** - Context 한도에 도달하지 않고 workflow가 모든 ticket을 원활하게 처리합니다.

4. **품질 유지** - 요약은 중요한 정보를 보존합니다.
   - 처리된 ticket과 ID
   - 지정된 범주와 우선순위
   - 전달된 팀
   - 전체 진행 상태
   
   모든 ticket은 여전히 제대로 분류되고, 우선순위가 지정되고, 전달되고, 응답됩니다.

5. **자연스러운 workflow** - 이는 실제 지원 agent가 일하는 방식을 반영합니다. Ticket을 해결하고 시스템에 간단히 기록한 다음 닫고 다음 ticket으로 이동합니다. 새로운 ticket을 처리하는 동안 모든 knowledge base 문서와 전체 응답 초안을 열어 두지는 않습니다.

Token 절감을 시각화하겠습니다.


In [8]:
# Baseline과 compaction 비교
print("=" * 70)
print("TOKEN USAGE COMPARISON")
print("=" * 70)
print(f"{'Metric':<30} {'Baseline':<20} {'With Compaction':<20}")
print("-" * 70)
print(f"{'Input tokens:':<30} {total_input:>19,} {total_input_compact:>19,}")
print(f"{'Output tokens:':<30} {total_output:>19,} {total_output_compact:>19,}")
print(
    f"{'Total tokens:':<30} {total_input + total_output:>19,} {total_input_compact + total_output_compact:>19,}"
)
print(f"{'Compactions:':<30} {'N/A':>19} {compaction_count:>19}")
print("=" * 70)

# 절감량 계산
token_savings = (total_input + total_output) - (total_input_compact + total_output_compact)
savings_percent = (
    (token_savings / (total_input + total_output)) * 100 if (total_input + total_output) > 0 else 0
)

print(f"\n💰 Token Savings: {token_savings:,} tokens ({savings_percent:.1f}% reduction)")

TOKEN USAGE COMPARISON
Metric                         Baseline             With Compaction     
----------------------------------------------------------------------
Input tokens:                              204,416              82,171
Output tokens:                               4,422               4,275
Total tokens:                              208,838              86,446
Compactions:                                   N/A                   2

💰 Token Savings: 122,392 tokens (58.6% reduction)


## Compaction의 내부 동작

`tool_runner`가 token 사용량이 threshold를 초과했음을 감지하면 자동으로 다음을 수행합니다.

1. 다음 API 호출 전에 **workflow를 일시 정지**합니다.
2. Claude에 진행 상황 요약을 요청하는 user message로 **요약 요청을 삽입**합니다.
3. **요약 생성** - Claude는 다음을 담은 요약을 `<summary></summary>` tag로 감싸 생성합니다.
   - **완료된 ticket**: 해결된 ticket의 간단한 기록(ID, 범주, 우선순위, 결과)
   - **진행 상태**: 처리한 ticket 수와 남은 ticket 수
   - **핵심 패턴**: Ticket 전반에서 두드러지는 추세
   - **다음 단계**: 다음에 할 일(남은 ticket 계속 처리)
4. **기록 지우기** - 전체 대화 기록(모든 tool 결과 포함)을 요약만으로 교체합니다.
5. **처리 재개** - Claude는 압축된 context로 다음 ticket 묶음을 처리하며 계속 작업합니다.


## Compaction 설정 사용자 정의

특정 사용 사례에 맞게 compaction 동작을 사용자 정의할 수 있습니다. 핵심 설정 옵션은 다음과 같습니다.

### Threshold 조정

`context_token_threshold`는 compaction이 실행되는 시점을 결정합니다.

```python
compaction_control={
    "enabled": True,
    "context_token_threshold": 5000,  # Ticket 5~7개 처리 후 압축
}
```

Threshold를 너무 낮게 설정하면 요약 자체가 compaction을 유발할 수 있습니다. 시연 목적으로 threshold를 5,000 token으로 설정했지만, 실제로는 다양한 설정을 실험하여 workflow에 가장 적합한 값을 찾으세요.

일반적인 지침은 다음과 같습니다.

- **낮은 threshold (5k-20k)**:
  - 명확한 경계가 있는 반복 작업 처리에 사용
  - 더 빈번한 compaction, 최소한의 context 누적
  - 순차적인 entity 처리에 가장 적합
  
- **중간 threshold (50k-100k)**:
  - 자연스러운 checkpoint가 더 적고 더 큰 여러 단계의 workflow
  - Context 유지와 관리 사이의 균형
  - Tool 호출 비용이 큰 workflow에 적합
  
- **높은 threshold (100k-150k)**:
  - 상당한 이전 context가 필요한 작업
  - Compaction이 덜 빈번하여 더 많은 원시 세부사항 보존
  - 호출당 비용은 높지만 compaction은 더 적음
  
- **기본값 (100k)**: 일반적인 장기 실행 작업에 적절한 균형

**Ticket 처리의 경우**: 각 ticket workflow가 상당한 tool 결과를 생성하지만 ticket은 독립적이므로 5k threshold가 잘 작동합니다. Ticket A를 해결한 후 ticket B를 처리할 때 A의 상세 KB 검색은 필요하지 않습니다.

### 요약에 다른 모델 사용하기

요약 생성에 더 빠르고 저렴한 모델을 사용할 수도 있습니다.

```python
compaction_control={
    "enabled": True,
    "model": "claude-haiku-4-5",  # 비용 효율적인 요약에 Haiku 사용
}
```


### 사용자 정의 요약 Prompt

요약 생성 방식을 유도하는 사용자 정의 prompt를 제공할 수 있습니다. 특정 종류의 정보를 보존해야 하는 고객 서비스 workflow에서 특히 유용합니다.

예를 들어 요구사항에 따라 사용자 정의 prompt를 정의할 수 있습니다.
- 완료된 모든 ticket의 **ticket 요약**
- 지정된 **범주와 우선순위**
- **전달된 팀**
- **진행 상태**(완료된 ticket, 남은 ticket)
- Workflow의 **다음 단계**

```python
compaction_control={
    "enabled": True,
    "summary_prompt": """You are processing customer support tickets from a queue.

Create a focused summary that preserves:

1. **COMPLETED TICKETS**: For each ticket you've fully processed:
   - Ticket ID and customer name
   - Issue category and priority assigned
   - Team routed to
   - Brief outcome

2. **PROGRESS STATUS**: 
   - How many tickets you've completed
   - Approximately how many remain in the queue

3. **NEXT STEPS**: Continue processing the next ticket

Format with clear sections and wrap in <summary></summary> tags."""
}
```


## Tool 없는 Compaction: 간단한 Chat Loop

위 예제는 tool을 많이 사용하는 agentic workflow에 초점을 맞추지만, context compaction은 사용자가 대화를 이끄는 **간단한 대화형 애플리케이션**에도 유용합니다.

 **참고:** 위에서 시연한 `compaction_control` parameter는 tool을 사용하는 agentic workflow의 `tool_runner`와 함께 동작합니다. Tool 없는 간단한 chat 애플리케이션에서는 같은 원칙으로 compaction을 직접 구현합니다.

사용자가 Claude와 복잡한 주제를 논의하고, 아이디어를 반복적으로 발전시키고, 문제를 해결하면서 긴 대화를 나누는 chat 애플리케이션을 생각해 보세요. 대화가 길어지면 동일한 context 누적 문제를 마주합니다.

**차이점**: Token 증가를 유발하는 것은 tool use 대신 대화 자체입니다. 각 대화 교환이 기록에 message를 추가합니다.
- 사용자가 질문합니다.
- Claude가 상세한 응답을 제공합니다.
- 사용자가 설명이나 상세 정보를 요청합니다.
- Claude가 더 많은 context로 응답합니다.
- 이를 수십 또는 수백 번 반복합니다.

Compaction 없이는 50번째 turn에서 모든 API 호출마다 전체 대화 기록(50번의 대화 교환 모두)을 보냅니다.

**해결책**: 같은 패턴으로 chat loop에 compaction을 직접 구현합니다.
1. 각 turn 후 token 사용량 추적
2. Threshold를 초과하면 요약 요청
3. 대화 기록을 요약으로 교체
4. 압축된 context로 대화 계속하기

구현 방법을 살펴보겠습니다.


In [ ]:
#!/usr/bin/env python3
"""
간단한 Compaction 예제 - 사용자가 이끄는 Chat Loop

Compaction이 있는 chat 애플리케이션의 기본 패턴을 보여줍니다.
Tool 없이 사용자가 계속할지 결정하는 간단한 loop만 있으면 됩니다.
"""

# 설정
COMPACTION_THRESHOLD = 3000  # Token이 이를 초과하면 압축(시연 목적의 낮은 값)

# Compaction을 위한 구조화된 요약 prompt
SUMMARY_PROMPT = """You have been working on the task described above but have not yet completed it. Write a continuation summary that will allow you (or another instance of yourself) to resume work efficiently in a future context window where the conversation history will be replaced with this summary. Your summary should be structured, concise, and actionable. Include:

1. **Task Overview**
   - The user's core request and success criteria
   - Any clarifications or constraints they specified

2. **Current State**
   - What has been completed so far
   - Files created, modified, or analyzed (with paths if relevant)
   - Key outputs or artifacts produced

3. **Important Discoveries**
   - Technical constraints or requirements uncovered
   - Decisions made and their rationale
   - Errors encountered and how they were resolved
   - What approaches were tried that didn't work (and why)

4. **Next Steps**
   - Specific actions needed to complete the task
   - Any blockers or open questions to resolve
   - Priority order if multiple steps remain

5. **Context to Preserve**
   - User preferences or style requirements
   - Domain-specific details that aren't obvious
   - Any promises made to the user

Be concise but complete—err on the side of including information that would prevent duplicate work or repeated mistakes.
 Write in a way that enables immediate resumption of the task.

Wrap your summary in <summary></summary> tags."""

# Message 기록
messages = []

print("Chat with Claude (type 'quit' to exit, or just hit Enter to continue)")
print("This is a demonstration - try having a conversation and watch compaction trigger")
print("=" * 60)

# 시연을 위한 대화 simulation
demo_messages = [
    "Help me understand how Python decorators work",
    "Can you show me an example with a timing decorator?",
    "How would I make a decorator that takes arguments?",
]

for user_input in demo_messages:
    print(f"\nYou: {user_input}")

    # User message 추가
    messages.append({"role": "user", "content": user_input})

    # Claude 응답 가져오기
    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        messages=messages,
    )

    messages.append(
        {
            "role": "assistant",
            "content": response.content,
        }
    )

    print("\nClaude: ", end="")
    for block in response.content:
        if block.type == "text":
            print(f"{block.text[:300]} ...")

    # 압축 여부 확인
    usage = response.usage

    # 총 token 계산(cache token 포함)
    total_input_tokens = (
        usage.input_tokens
        + (usage.cache_creation_input_tokens or 0)
        + (usage.cache_read_input_tokens or 0)
    )
    total_tokens = total_input_tokens + usage.output_tokens

    cache_info = ""
    if usage.cache_creation_input_tokens or usage.cache_read_input_tokens:
        cache_info = f" (cache: {usage.cache_creation_input_tokens or 0} write + {usage.cache_read_input_tokens or 0} read)"

    print(
        f"\n[Tokens: {total_input_tokens} in{cache_info} + {usage.output_tokens} out = {total_tokens} total]"
    )

    if total_tokens > COMPACTION_THRESHOLD:
        print(f"\n{'=' * 60}")
        print(f"🔄 Compacting conversation... {len(messages)} messages → ", end="", flush=True)

        # 구조화된 prompt로 요약 받기
        summary_response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            messages=messages + [{"role": "user", "content": SUMMARY_PROMPT}],
        )

        summary_text = "".join(
            block.text for block in summary_response.content if block.type == "text"
        )

        # 기록을 요약으로 교체
        messages = [{"role": "user", "content": summary_text}]

        print("1 message")
        print(f"{'=' * 60}\n")

print(f"Final conversation messages: {messages[-1].get('content')}")

print("\nDemo complete! In a real application, this loop would continue with user input.")

Chat with Claude (type 'quit' to exit, or just hit Enter to continue)
This is a demonstration - try having a conversation and watch compaction trigger

You: Help me understand how Python decorators work


### Chat Loop 패턴 이해하기

위 예제는 대화 context에서 수동 compaction을 보여줍니다. 동작 방식은 다음과 같습니다.

**핵심 구성 요소:**

1. **Token 추적**: 각 응답 후 총 token(input + output + cache token) 계산
2. **Threshold 확인**: 총합이 threshold를 초과하면 compaction 실행
3. **요약 요청**: 같은 구조화된 SUMMARY_PROMPT를 Claude에 전송
4. **기록 교체**: 전체 message 기록을 요약만으로 교체
5. **계속하기**: 다음 user message는 전체 기록 대신 요약을 기반으로 함

**이 패턴을 사용할 때:**

- **긴 brainstorming session**: 사용자가 많은 turn에 걸쳐 Claude와 아이디어 탐색
- **학습 대화**: 수십 번의 교환에 걸친 tutorial이나 설명
- **반복적 개선**: 사용자가 초안, 설계, 해결책에 feedback 제공
- **Chat 애플리케이션**: 모든 multi-turn 대화 interface

**Tool Runner와의 핵심 차이점:**

| 측면 | Tool Runner (자동) | Chat Loop (수동) |
|--------|------------------------|-------------------|
| **실행 조건** | Threshold 도달 시 자동 | 직접 threshold 확인 구현 |
| **요약** | SDK가 요약 요청 처리 | 명시적으로 API 호출 |
| **기록 관리** | SDK가 message 교체 | 수동으로 list 교체 |
| **사용 사례** | Tool을 사용하는 agentic workflow | 사용자가 이끄는 대화 |

**Production 고려사항:**

1. **Threshold 조정**: 실제 애플리케이션에서는 더 큰 threshold 사용
2. **요약 prompt 사용자 정의**: 대화 유형에 맞게 조정(brainstorming, 기술 지원, tutoring)
3. **사용자 표시 제공**: "대화 요약 중..."과 같은 message를 표시하여 사용자가 잠시 멈추는 이유를 이해하도록 함
4. **핵심 context 보존**: 요약 prompt가 사용자가 중요하게 여기는 도메인별 정보를 포착하도록 함

이 패턴은 compaction의 시점과 방식을 완전히 제어할 수 있어, SDK의 automatic tool-runner compaction을 사용할 수 없는 대화형 애플리케이션에 이상적입니다.


## 한계와 고려사항

Automatic context compaction은 강력하지만 알아야 할 중요한 한계가 있습니다.

### Server-Side Sampling Loops

**현재 한계**: Compaction은 server-side web search tool 같은 server-side sampling loop에서 최적으로 동작하지 않습니다.

**이유**: Cache token이 sampling loop에 걸쳐 누적되어 실제 대화 기록 대신 cache된 내용을 기준으로 compaction이 너무 일찍 실행될 수 있습니다.

이 기능은 다음과 함께 가장 잘 동작합니다.
- ✅ Client-side tool(이 cookbook의 고객 서비스 API 등)
- ✅ 정기적인 tool use가 있는 표준 agentic workflow
- ✅ 파일 작업, database query, API 호출
- ❌ Server-side Extended Thinking
- ❌ Server-side web search tool

### 정보 손실

**Trade-off**: 요약은 본질적으로 일부 정보를 잃습니다. Claude는 핵심을 잘 식별하지만 일부 세부사항은 압축되거나 생략됩니다.

**Ticket 처리에서:**
- ✅ **유지됨**: Ticket ID, 범주, 우선순위, 팀, 결과, 진행 상태
- ❌ **손실됨**: 전체 knowledge base 문서 텍스트, 완전한 응답 초안 텍스트, 상세한 분류 추론

보통 이는 허용 가능합니다. 모든 KB 문서와 전체 응답 텍스트를 영구히 보관할 필요는 없고 완료 기록만 있으면 됩니다.

**완화 방법:**
- 사용자 정의 요약 prompt로 중요한 정보 보존
- 광범위한 이전 context가 필요한 작업에는 더 높은 threshold 설정
- 작업을 module 단위로 구성(각 단계가 원시 세부사항 대신 요약을 기반으로 함)

### Compaction을 사용하지 말아야 할 때

다음에는 compaction을 피하세요.

1. **짧은 작업**: 작업이 50k-100k token 내에서 완료되면 compaction은 불필요한 overhead를 추가합니다.
2. **전체 audit trail이 필요한 작업**: 일부 작업은 이전의 모든 세부사항에 접근해야 합니다.
3. **Server-side sampling workflow**: 위에서 언급했듯이 이 한계가 해결될 때까지 기다리세요.
4. **반복적 개선이 매우 많은 경우**: 각 단계가 앞선 모든 단계의 정확한 세부사항에 결정적으로 의존하는 작업

### Compaction을 사용해야 할 때

Compaction은 다음에 이상적입니다.

1. **순차 처리**: Ticket workflow처럼 여러 항목을 하나씩 처리
2. **여러 단계의 workflow**: 각 단계가 다음으로 넘어가기 전에 진행 상황 요약 가능
3. **반복적 데이터 처리**: 큰 dataset을 chunk 단위 또는 entity 하나씩 처리
4. **긴 분석 session**: 많은 entity에 걸쳐 데이터 분석
5. **Batch 작업**: 각각 독립적인 수백 개의 항목 처리

**Ticket 처리는 완벽한 사용 사례**입니다. 그 이유는:
- 각 ticket workflow가 대부분 독립적입니다.
- 전체 tool 결과 대신 완료 요약이 필요합니다.
- 자연스러운 compaction 지점이 있습니다(여러 ticket 완료 후).
- Workflow가 반복적이고 순차적입니다.


## 요약

Automatic context compaction은 장기 실행 agentic workflow가 일반적인 context 한도를 넘을 수 있게 하는 강력한 기능입니다. 이 cookbook에서는 고객 서비스 ticket 처리 workflow를 통해 compaction을 살펴보았습니다.

### 다음 단계

자신의 workflow에 compaction을 구현해 보세요.
1. 자연스러운 compaction 지점 파악(각 항목 처리 후, 각 단계 완료 후 등)
2. 항목별 경계가 명확하면 적극적인 threshold(5k-10k)로 시작
3. 사용자 정의 요약 prompt로 중요한 정보 보존
4. Compaction 실행 시점을 모니터링하고 품질 유지 여부 확인
5. 특정 요구에 따라 threshold 조정

효과적인 context 관리에 대해서는 [Effective Context Engineering for AI Agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)를 참조하세요.
